In [5]:
import torch
import torch.nn as nn

### 2. Model Architecture

In [6]:
# Darknet Architectures Configuration
architectures_config = [
    # Tuple: (kernel_size, num_filters, stride, padding)
    (7, 64, 2, 3),
    "M", # MaxPool 2x2
    (3, 192, 1, 1),
    "M",
    (1, 128, 1, 0),
    (3, 256, 1, 1),
    (1, 256, 1, 0),
    (3, 512, 1, 1),
    "M",
    [(1, 256, 1, 0), (3, 512, 1, 1), 4],
    (1, 512, 1, 0),
    (3, 1024, 1, 1),
    "M",
    # List: Tuple 1, Tuple 2, Number repeat
    [(1, 512, 1, 0), (3, 1024, 1, 1), 2],
    (3, 1024, 1, 1),
    (3, 1024, 2, 1),
    (3, 1024, 1, 1),
    (3, 1024, 1, 1),
]

In [7]:
### CNNBLOCK ###
# X --> Z = W * X + b --> Z_batch = BatchNorm(Z) --> A = LeakyReLU(Z_batch)
def CNNBlock(in_channels, out_channels, kernel_size, stride, padding):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding),
        nn.BatchNorm2d(out_channels),
        nn.LeakyReLU(0.1)
    )

### FULLY CONNECTED LAYER ###
def create_fcs(split_size=7, num_boxes=2, num_classes=20):
    S, B, C = split_size, num_boxes, num_classes
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(1024 * S * S, 496),
        nn.LeakyReLU(0.1),
        nn.Dropout(0.5),
        nn.Linear(496, S * S * (B * 5 + C))
    )

In [8]:
### CREATE CONV LAYER ###
def create_conv_layer(in_channels, architectures):
    layers = []
    
    for x in architectures:
        if type(x) == tuple:
            layers += [CNNBlock(in_channels, x[1], kernel_size=x[0], stride=x[2], padding=x[3])]
            in_channels = x[1]

        elif type(x) == str:
            layers += [nn.MaxPool2d(kernel_size=2, stride=2)]

        elif type(x) == list:
            conv1 = x[0]
            conv2 = x[1]
            num_repeats = x[2]

            for _ in range(num_repeats):
                layers += [CNNBlock(in_channels, conv1[1], kernel_size=conv1[0], stride=conv1[2], padding=conv1[3])]
                in_channels = conv1[1]
                
                layers += [CNNBlock(conv1[1], conv2[1], kernel_size=conv2[0], stride=conv2[2], padding=conv2[3])]
                in_channels = conv2[1]

    return nn.Sequential(*layers)

In [ ]:
### YOLOv1 MODEL ###
def YOLOv1(in_channels=3, split_size=7, num_boxes=2, num_classes=20):
    # Backbone
    darknet_backbone = create_conv_layer(in_channels, architectures_config)

    # Head
    fcs_head = create_fcs(split_size, num_boxes, num_classes)
    
    # Model
    model = nn.Sequential(
        darknet_backbone,
        fcs_head
    )

    return model

torch.Size([2, 1470])
